### Pseudonymization Strategy & GDPR Compliance

To secure the dataset, we applied cryptographic hashing (pseudonymization) to direct identifiers such as `.ssn` , `.email` , `.full_name`,`.ip_address`an `zip_code`. 

**Why we chose this approach and why it is legally sufficient:**

* **Preserving Data Utility:** Unlike full data deletion or simple masking, pseudonymization protects user privacy while allowing the Data Science team to track unique applicant records. This is essential for accurately calculating fairness metrics and auditing the credit model.
* **Data Minimization & Security (Art. 5 & Art. 32):** By transforming raw PII into hashed values, we ensure that sensitive personal data is no longer stored without protection, preventing exposure in plain text to analysts or downstream systems.
* **Right to Erasure (Art. 17):** This approach elegantly handles the "Right to be Forgotten". By using a secret cryptographic key (a "salt") during the hashing process, we can simply delete the key when a user requests data erasure. This process, known as *crypto-shredding*, irreversibly turns the pseudonymized data into fully anonymized data, It satisfies GDPR Article 17 requirements perfectly, without forcing us to destroy valuable historical statistical data.

While techniques like **k-anonymity** and **differential privacy** provide strong mathematical guarantees against re-identification, they often require aggregating or adding noise to the data, which degrades its utility for individual-level credit risk modeling. **Tokenization** requires maintaining a secure lookup table, creating a single point of failure. We chose SHA-256 Hashing with a **Salt**  because it allows us to maintain referential integrity (tracking the same user across datasets) without storing a vulnerable lookup table, while still enabling GDPR Article 17 compliance via crypto-shredding.

In [9]:
import os
import pandas as pd
import hashlib
import json

# 1. Load the nested JSON data
with open('../data/raw_credit_applications.json', 'r') as file:
    data = json.load(file)

df = pd.json_normalize(data)

# 2. Define the pseudonymization function
os.environ['SALT'] = "NovaCred_Secure_2026!"

def pseudonymize_pii(value):
    if pd.isna(value) or value == "":
        return value
    
    # Combine the value with the pseudonym and encode
    salted_value = str(value) + os.environ['SALT']
    
    # Return the SHA-256 hexadecimal hash
    return hashlib.sha256(salted_value.encode()).hexdigest()

# 3. Apply the function to the PII columns
# Targeting the nested fields from the schema
df['applicant_info.ssn_hashed'] = df['applicant_info.ssn'].apply(pseudonymize_pii)
df['applicant_info.email_hashed'] = df['applicant_info.email'].apply(pseudonymize_pii)
df['applicant_info.full_name_hashed'] = df['applicant_info.full_name'].apply(pseudonymize_pii)
df['applicant_info.ip_address_hashed'] = df['applicant_info.ip_address'].apply(pseudonymize_pii)
df['applicant_info.zip_code_hashed'] = df['applicant_info.zip_code'].apply(pseudonymize_pii)
# 4. Drop the original raw PII columns to ensure data minimization
df = df.drop(columns=['applicant_info.ssn', 'applicant_info.email', 'applicant_info.full_name', 'applicant_info.ip_address', 'applicant_info.zip_code'])

# 5. Verify the transformation
print(df[['_id', 'applicant_info.ssn_hashed', 'applicant_info.email_hashed','applicant_info.full_name_hashed', 'applicant_info.ip_address_hashed', 'applicant_info.zip_code_hashed']].head())

       _id                          applicant_info.ssn_hashed  \
0  app_200  9196537eb3af63e06878de29f8e1c7d3897791fe6032e1...   
1  app_037  f7e04e392a2676c1008fc9dc21750b3176c57c5eed8809...   
2  app_215  3263f19e841649667a994a137d9d2444feae15e1b50417...   
3  app_024  496b136660dfbe2bf483a50420f917e89e49e666a0ae79...   
4  app_184  5f688fde89897de7828b5eddf85be3ab066b2d9a5066da...   

                         applicant_info.email_hashed  \
0  43816fd1eb76c24869e6e312699cda3d95ff057122fbfb...   
1  5f4a8dd0e06f28d96333869faf36bc4b38ba9727c68526...   
2  63be22cf9717bb8f86a2800acae73c429d08b8246ea6a3...   
3  807aef6fb2d50f66c53a38597b8e437b955923c8636547...   
4  6bead4f1f8650f583ff89c06a5697b6aab4a8c20db519c...   

                     applicant_info.full_name_hashed  \
0  070c9881585996a6be0283cff64d93979b3a82736e602e...   
1  5818e748b1730f6f2e23c72eb1db5b19a09edf7207351e...   
2  eb51c1b818060437741f3718afcc89d6aa27c7c309a7c7...   
3  a5a462d3cab268cf103c86d825b6e69a5c01756634f28

### Privacy Demonstration: GDPR Article 17 (Right to Erasure) & Key Management

To fully comply with GDPR requirements, specifically the right to erasure, we must be able to securely delete a user's personal data upon request. 

**Why we use Environment Variables (`os.environ`):**
Storing a cryptographic salt in plaintext within a notebook or script is a severe key management failure. In a real-world production environment, this secret must be managed via a secure vault (e.g., AWS KMS) and injected into the application at runtime. We simulate this secure architecture here by managing the salt exclusively through environment variables (`os.environ`).

**The Crypto-Shredding Process:**
When a user (e.g., `app_001`) exercises their Right to be Forgotten:
1. We do not need to manually search and destroy every individual record across our distributed databases.
2. Instead, we perform **crypto-shredding** by permanently deleting the central secret salt.
3. Once the salt is destroyed from the environment, the SHA-256 hashes become mathematically irreversible. This instantly and effectively transforms the pseudonymized data into fully anonymized data[cite: 142], completely satisfying the Article 17 request while preserving the dataset's statistical utility.

In [10]:
import os
# 1. User app_001 requests deletion (Article 17)
# 2. To comply, we "crypto-shred" by deleting the centralized secret salt/pepper
del os.environ['SALT'] 

# 3. Without the salt, the hashes cannot be reversed or regenerated via brute force
print("Salt destroyed. Data is now functionally anonymized.")

Salt destroyed. Data is now functionally anonymized.
